# DSA 210 — Tourism Demand Under Shock
## Eren Sean Harley | 36054 | Sabancı University, Spring 2026

---

## Project Overview

Turkey is one of the world's most geographically and politically exposed tourism markets. Sitting at the crossroads of Europe, the Middle East, and the former Soviet sphere, it has absorbed shocks ranging from the 2016 coup attempt to the Syrian civil war to COVID-19 — all within two decades. In 2019, Turkey welcomed over 51 million foreign visitors, making it a top-5 global destination. That number collapsed to 15 million in 2020.

This project asks a question that tourism firms, policymakers, and researchers all care about: **which source markets absorb shocks and bounce back, and which ones don't?** More specifically, with MENA tensions rising again since 2023, should Turkish tourism firms be worried about losing European visitors, Gulf visitors, or Russian visitors — and what does the data actually say?

I analyze 17 source markets across 23 years (2003–2025) using visitor count data from TÜİK (Turkish Statistical Institute), macro indicators from the IMF, and the World Bank Political Stability Index. The analysis is organized around three deliverables:

1. **Market sensitivity** — which markets over- or under-react to each shock type (coup, regional conflict, pandemic, currency)?
2. **Recovery rates** — how fast does each market bounce back, and does recovery speed differ by geography or market type?
3. **Forward-looking targeting** — given rising MENA tensions, which markets should tourism firms prioritize?

---

*AI assistance (Claude, Anthropic) was used for code review, data pipeline structuring, and repository cleanup, in accordance with the DSA 210 academic integrity policy.*

## 1. Setup & Data Loading

Before any analysis, I load all necessary libraries and the merged panel dataset. The panel is the single source of truth for all subsequent analysis — it was constructed by `notebooks/01_data_pipeline.py`, which merges TÜİK visitor counts with IMF macro indicators and the World Bank Political Stability Index across 17 countries and 23 years (2003–2025).

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Paths (notebook lives in notebooks/, data/images are one level up) ────
NB_DIR   = Path.cwd()
ROOT_DIR = NB_DIR.parent
DATA_DIR = ROOT_DIR / 'data'
IMG_DIR  = ROOT_DIR / 'images'

# ── Load panel ────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'panel_dataset.csv')
all_countries = sorted(df['country'].unique())

print(f"Panel shape: {df.shape}")
print(f"Countries ({len(all_countries)}): {', '.join(all_countries)}")
print(f"Years: {df['year'].min()} – {df['year'].max()}")
df.head()

### What's in the panel?

Each row is a **country × year** observation. The key columns are:

| Column | Description |
|---|---|
| `country`, `year` | Market identifier and year |
| `visitors` | Annual visitors from that market to Turkey |
| `log_visitors` | log(visitors) — used as the regression target |
| `market_group` | Western Europe / Eastern Europe / Former Soviet / MENA / Other |
| `gdp_growth`, `gdp_per_capita`, `inflation` | IMF macro indicators for the source country |
| `political_stability` | World Bank WGI Political Stability index (source country) |
| `tur_ppp_rate`, `tur_currency_weakness` | Turkey macro: PPP exchange rate and its YoY % change |
| `coup_2016`, `covid`, `syria_conflict`, ... | Binary shock dummies (1 = shock active in that year) |

A few known quirks: Syria has zero or near-zero visitors from 2011 onward (the country became a conflict origin, not a tourist-sending market). COVID dummies fire for 2020–2021. The `mena_tension_recent` dummy fires from 2023 onward.

In [ ]:
# Quick data audit
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print("Market groups:")
print(df.groupby('market_group')['country'].unique())
print()
print("Visitor range (millions):")
print((df['visitors'] / 1e6).describe().round(2))

## 2. Per-Country Overview

Before running any aggregate tests, I generated individual diagnostic pages for all 17 markets — each showing visitor trends alongside the source country's GDP growth, inflation, and political stability. This is essential groundwork: you cannot run meaningful hypothesis tests without first understanding which markets have structural breaks, data gaps, or unusual shapes.

Below are the key summary figures produced by `notebooks/00_per_country_eda.py`.

In [ ]:
from IPython.display import Image, display

print("=== 17-Country Visitor Grid ===")
display(Image(filename=str(IMG_DIR / 'per_country' / '_grid_visitors.png'), width=900))

**What I see in the visitor grid:**

The 17-country grid immediately reveals several structural patterns. Western European markets (France, Germany, UK, Netherlands) are large and stable — they dominate aggregate visitor counts but show predictable shock responses. Former Soviet markets (Russia, Azerbaijan, Georgia) are volatile: Russia had a sharp tourist ban in 2015–2016 (jet crisis) before partially recovering. MENA markets tell a split story: conflict-affected markets (Syria, Iran, Iraq, Israel) are structurally suppressed, while Gulf markets (UAE, Qatar) look more like Western European markets in trajectory. The USA is a small, steady market.

COVID (2020–2021) creates a universal crash visible across all 17 panels. The recovery shape varies dramatically — some markets were back at baseline by 2022, others are still below their 2019 levels in 2025.

In [ ]:
print("=== Turkey Macro Context ===")
display(Image(filename=str(IMG_DIR / 'per_country' / '_turkey_macro.png'), width=800))

**Turkey's macro context:**

Turkey's own economic environment matters for inbound tourism through two main channels: the PPP exchange rate (a weaker lira makes Turkey cheaper for foreign visitors) and political stability (instability deters visitors regardless of price). The macro context chart shows Turkey's political stability index declining steadily from 2003 to 2016 (coup attempt), recovering modestly, then dipping again post-2022 as MENA tensions rose. The lira has weakened dramatically since 2018, which is a theoretical tailwind for demand — but large currency crises (2018, 2021–2022) tend to coincide with broader instability that offsets the price advantage.

## 3. Perspective EDA — Five Key Aggregate Views

Before testing specific hypotheses, I built five aggregate figures that set the context and guide the hypothesis design. These were generated by `notebooks/03_perspective_eda.py` using `data/panel_dataset.csv`.

In [ ]:
print("EDA Figure 1: Visitor Trends by Market Group (2003–2025)")
display(Image(filename=str(IMG_DIR / 'eda_01_trends_by_group.png'), width=850))

**Trends by group:** Western Europe is the dominant source region but its growth has slowed post-2016. Former Soviet markets grew fastest pre-COVID, driven mainly by Russian visitors, but the Russia-Turkey jet crisis (2015) and Russia-Ukraine war (2022) created structural breaks. MENA markets have a complex shape: early growth through the 2000s, suppression during the Syrian war period (2011–2015), and a split post-2023 between stable Gulf markets and conflict-affected markets.

In [ ]:
print("EDA Figure 2: Shock Sensitivity by Country and Shock Type")
display(Image(filename=str(IMG_DIR / 'eda_02_shock_sensitivity.png'), width=850))

**Shock sensitivity:** The heatmap-style sensitivity figure reveals that COVID produced by far the most uniform and severe response across all markets — nearly every country saw a 70-90% drop. The 2016 coup was more heterogeneous: some markets barely moved (Georgia, Iraq), while others fell sharply (Germany, Netherlands). The Syria conflict period was unusual in that several MENA markets actually grew (Iraq, notably), because the conflict redirected regional travel patterns toward Turkey as a safe hub.

In [ ]:
print("EDA Figure 3: Post-COVID Recovery Speed")
display(Image(filename=str(IMG_DIR / 'eda_03_recovery_speed.png'), width=850))

**Recovery speed:** Western European markets generally recovered fastest (by 2022 for most), while Former Soviet markets were slower (Russia permanently displaced by 2022 sanctions, Georgia recovering well). Several MENA markets — particularly conflict-affected ones — had not recovered to 2019 levels by 2025, as the ongoing wars in the region suppressed outbound tourism capacity.

In [ ]:
print("EDA Figure 4: Turkey Political Stability Index (2003–2024)")
display(Image(filename=str(IMG_DIR / 'eda_04_political_stability.png'), width=750))

In [ ]:
print("EDA Figure 5: Country × Year Resilience Heatmap")
display(Image(filename=str(IMG_DIR / 'eda_05_resilience_heatmap.png'), width=850))

**Resilience heatmap:** Viewing visitor volumes as a percentage of each market's own baseline (rather than raw counts) reveals a striking pattern: the 2020 COVID crash is universal and the deepest in 22 years, but the recovery variance across countries is enormous. UAE and Qatar show strong above-baseline performance by 2023–2025. Conflict-affected MENA markets remain persistently below baseline. This heatmap directly motivates the within-MENA descriptive split in H6.

## 4. Hypothesis Tests (H1–H6)

All six hypothesis tests follow the lecture-aligned plan from CLAUDE.md:

- **H1, H3** — One-way ANOVA + Kruskal-Wallis robustness check (Tukey HSD deliberately excluded — not in lecture scope)
- **H2, H6** — Levene's test for equal variances → independent t-test if equal, Mann-Whitney U fallback if unequal
- **H4** — Pearson correlation on `tur_currency_weakness` (YoY %, not the PPP level)
- **H5** — Pearson + Spearman aggregate, plus per-group Spearman breakdown

I run each test from scratch below using `data/panel_dataset.csv` as the single source of truth.

In [ ]:
# ── Shared helpers (used across H1-H6) ──────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#e0e0e0', 'grid.linewidth': 0.8,
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 10, 'legend.fontsize': 9,
})

GROUP_COLORS = {
    'Western Europe': '#2196F3', 'Eastern Europe': '#4CAF50',
    'Former Soviet':  '#FF5722', 'MENA':          '#9C27B0',
    'Other':          '#607D8B',
}

def shock_impact(country, shock_year, pre_window=2):
    d   = df[df['country'] == country].sort_values('year')
    pre = d[d['year'].between(shock_year - pre_window, shock_year - 1)]['visitors'].mean()
    sv  = d[d['year'] == shock_year]['visitors'].values
    if pre == 0 or len(sv) == 0 or np.isnan(pre):
        return np.nan
    return (sv[0] / pre - 1) * 100

def years_to_recover(country, shock_year, pre_window=2):
    d   = df[df['country'] == country].sort_values('year')
    pre = d[d['year'].between(shock_year - pre_window, shock_year - 1)]['visitors'].mean()
    if np.isnan(pre) or pre == 0:
        return np.nan
    for _, row in d[d['year'] > shock_year].iterrows():
        if row['visitors'] >= pre:
            return row['year'] - shock_year
    return np.nan

def levene_then_test(group_a, group_b, label_a='A', label_b='B'):
    lev_stat, lev_p = stats.levene(group_a, group_b)
    equal_var = lev_p > 0.05
    if equal_var:
        main_stat, main_p = stats.ttest_ind(group_a, group_b, equal_var=True)
        test_name  = "Independent t-test (equal variances)"
        stat_label = f"t={main_stat:.3f}"
    else:
        main_stat, main_p = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')
        test_name  = "Mann-Whitney U (unequal variances)"
        stat_label = f"U={main_stat:.1f}"
    print(f"  Levene: W={lev_stat:.3f}, p={lev_p:.4f} → {'equal var → t-test' if equal_var else 'unequal var → Mann-Whitney U'}")
    print(f"  {test_name}: {stat_label}, p={main_p:.4f} → {'SIGNIFICANT ✓' if main_p < 0.05 else 'not significant'} at α=0.05")
    print(f"  {label_a} mean: {group_a.mean():+.1f}%  (n={len(group_a)})")
    print(f"  {label_b} mean: {group_b.mean():+.1f}%  (n={len(group_b)})")
    return {'lev_stat': lev_stat, 'lev_p': lev_p, 'equal_var': equal_var,
            'test_name': test_name, 'stat_label': stat_label,
            'main_stat': main_stat, 'main_p': main_p,
            'significant': main_p < 0.05}

print("Helpers loaded. Panel shape:", df.shape)

### H1 — 2016 Coup Sensitivity by Market Group

**Null hypothesis (H₀):** All market groups respond equally to Turkey's 2016 coup attempt.

**Alternative (H₁):** At least one market group responds differently.

**Test:** One-way ANOVA (primary) + Kruskal-Wallis (robustness check). The "Other" group (USA only, n=1) is excluded from both tests — you need at least n=2 per group for ANOVA/KW to be valid. Tukey HSD post-hoc is deliberately excluded as it is not taught in the DSA 210 lectures; any interpretation of which specific group is highest is therefore descriptive only, not inferential.

In [ ]:
# H1: 2016 coup sensitivity
coup_impact = {c: shock_impact(c, 2016, pre_window=2) for c in all_countries}
coup_df = pd.DataFrame.from_dict(coup_impact, orient='index', columns=['impact'])
coup_df.dropna(inplace=True)
coup_df['group'] = coup_df.index.map(lambda c: df[df['country']==c]['market_group'].values[0])

group_order_h1 = ['Western Europe', 'Eastern Europe', 'Former Soviet', 'MENA']
groups_data_h1 = {g: coup_df[coup_df['group']==g]['impact'].values for g in group_order_h1}

print("Mean % change in 2016 vs 2014-15 baseline, by group:")
for g, vals in groups_data_h1.items():
    if len(vals):
        print(f"  {g:<22}: {vals.mean():+.1f}%  (n={len(vals)})")
print("  Other (USA, n=1): excluded from tests")

valid_h1 = [v for v in groups_data_h1.values() if len(v) >= 2]
f1_stat, p1_anova = stats.f_oneway(*valid_h1)
h1_kw_stat, p1_kw = stats.kruskal(*valid_h1)

print(f"\nOne-way ANOVA:    F={f1_stat:.3f}, p={p1_anova:.4f} → {'SIGNIFICANT ✓' if p1_anova<0.05 else 'not significant'}")
print(f"Kruskal-Wallis:   H={h1_kw_stat:.3f}, p={p1_kw:.4f} → {'SIGNIFICANT ✓' if p1_kw<0.05 else 'not significant'}")

In [ ]:
display(Image(filename=str(IMG_DIR / 'h1_coup_sensitivity.png'), width=850))

**Interpretation:** The ANOVA and Kruskal-Wallis results tell us whether the four market groups differ in aggregate sensitivity to the 2016 coup. The p-values determine whether we can claim a statistically significant group-level difference. If non-significant (as the README finds: ANOVA F=0.75, p=0.54; KW H=3.24, p=0.36), we cannot claim any group is more sensitive than another — even though Western Europe shows the largest descriptive mean drop. High within-group variance (e.g., Russia dropped sharply while Azerbaijan barely moved, both in "Former Soviet") explains why the omnibus test fails to find significance despite apparent mean differences.

### H2 — Syria Conflict: Did Geographic Proximity Predict Visitor Change?

**Null hypothesis (H₀):** Syria-bordering markets (Iraq, Iran, Israel) changed by the same amount as non-bordering markets during the Syrian war period (2011–2015 vs 2008–2010).

**Alternative (H₁):** Syria-bordering markets changed differently.

**Test:** Levene's test for equal variances → independent t-test or Mann-Whitney U. Syria itself is excluded from the test — it IS the conflict origin country, not a comparator. The 2008–2010 window was a relative tourism trough for Turkey, so most countries show absolute gains over 2011–2015; the test is about *relative* differences between bordering and non-bordering groups.

In [ ]:
# H2: Syria conflict proximity
SYRIA_BORDERING = ['Iraq', 'Iran', 'Israel']
test_countries_h2 = [c for c in all_countries if c != 'Syria']

pre_syria  = (df[(df['year'].between(2008,2010)) & df['country'].isin(test_countries_h2)]
              .groupby(['country','market_group'])['visitors'].mean())
dur_syria  = (df[(df['year'].between(2011,2015)) & df['country'].isin(test_countries_h2)]
              .groupby(['country','market_group'])['visitors'].mean())
syria_chg  = ((dur_syria - pre_syria) / pre_syria * 100).reset_index()
syria_chg.columns = ['country', 'market_group', 'pct_change']
syria_chg['proximity'] = syria_chg['country'].apply(
    lambda c: 'Syria-bordering' if c in SYRIA_BORDERING else 'Non-bordering')
syria_chg.dropna(subset=['pct_change'], inplace=True)

bordering_chg     = syria_chg[syria_chg['proximity']=='Syria-bordering']['pct_change'].values
non_bordering_chg = syria_chg[syria_chg['proximity']=='Non-bordering']['pct_change'].values

print("H2 — Levene + t-test / Mann-Whitney U:")
h2_result = levene_then_test(bordering_chg, non_bordering_chg,
                              label_a='Syria-bordering', label_b='Non-bordering')

In [ ]:
display(Image(filename=str(IMG_DIR / 'h2_syria_proximity.png'), width=900))

**Interpretation:** A non-significant result here is actually the most interesting finding. Iraq grew 160% during the Syria war period (2011–2015 vs 2008–2010), seemingly because the Syrian conflict redirected regional travel toward Turkey and the Gulf as safe hubs. This masks any aggregate suppression effect from the bordering group, producing a p-value well above 0.05. The lesson: raw geographic proximity is not a reliable predictor when conflict redirects rather than suppresses mobility. The non-significant result is honest and defensible.

### H3 — COVID Recovery Speed by Market Group

**Null hypothesis (H₀):** All market groups recover from COVID at the same speed.

**Alternative (H₁):** Recovery speed differs across groups.

**Test:** One-way ANOVA + Kruskal-Wallis. "Years to recover" is defined as the number of years after 2020 until a market's visitor count first exceeds the 2018–2019 mean. Countries that have not recovered by 2025 are coded as unrecovered and excluded from the test (their exclusion is reported transparently).

In [ ]:
# H3: COVID recovery speed
rec_years_h3 = {c: years_to_recover(c, 2020, pre_window=2) for c in all_countries}
rec_df = pd.DataFrame.from_dict(rec_years_h3, orient='index', columns=['years_to_recover'])
rec_df['group'] = rec_df.index.map(lambda c: df[df['country']==c]['market_group'].values[0])

not_rec_h3 = rec_df[rec_df['years_to_recover'].isna()].index.tolist()
print(f"Not recovered by 2025 (excluded from test): {not_rec_h3}")

rec_groups = {g: rec_df[rec_df['group']==g]['years_to_recover'].dropna().values
              for g in ['Western Europe','Eastern Europe','Former Soviet','MENA']}
print("\nMean recovery years (recovered countries only):")
for g, vals in rec_groups.items():
    if len(vals):
        print(f"  {g:<22}: {vals.mean():.1f} yrs  (n={len(vals)})")

valid_rec = [v for v in rec_groups.values() if len(v) >= 2]
if len(valid_rec) >= 2:
    f3_stat, p3_anova = stats.f_oneway(*valid_rec)
    h3_kw_stat, p3_kw = stats.kruskal(*valid_rec)
    print(f"\nOne-way ANOVA:    F={f3_stat:.3f}, p={p3_anova:.4f} → {'SIGNIFICANT ✓' if p3_anova<0.05 else 'not significant'}")
    print(f"Kruskal-Wallis:   H={h3_kw_stat:.3f}, p={p3_kw:.4f} → {'SIGNIFICANT ✓' if p3_kw<0.05 else 'not significant'}")
    print("Note: small effective n per group — tests have low power")

In [ ]:
display(Image(filename=str(IMG_DIR / 'h3_covid_recovery.png'), width=900))

**Interpretation:** The small multiples show that recovery trajectories differ visually — Western Europe and some MENA markets bounced back quickly, while Former Soviet markets were disrupted by the Russia-Ukraine war (Russia, which was the dominant market in the Former Soviet group, lost ~80% of its Turkey visits after 2022 sanctions). However, the small effective n per group (after excluding unrecovered countries) reduces test power substantially. A non-significant result (ANOVA p=0.18 per README) is therefore expected and honest — it does not mean recovery speeds are the same, only that we cannot detect the difference with this sample size.

### H4 — Does Turkey Lira Weakness Predict Visitor Drops During Shocks?

**Null hypothesis (H₀):** Turkey lira weakness (YoY % change in PPP rate) is not correlated with visitor drops during shock years.

**Alternative (H₁):** Larger lira weakness correlates with larger visitor drops.

**Test:** Pearson correlation. The variable is `tur_currency_weakness` — the year-over-year percentage change in Turkey's PPP exchange rate, NOT the PPP level itself. The level rises monotonically and would conflate "weaker lira" with "later in time." I use 17 countries × 3 shock years (2009, 2016, 2020) = 51 observations.

In [ ]:
# H4: Lira weakness vs visitor drops
shock_years_h4 = [2009, 2016, 2020]
results_h4 = []
for c in all_countries:
    for sy in shock_years_h4:
        imp = shock_impact(c, sy, pre_window=2)
        tur_row = df[df['year']==sy][['tur_currency_weakness']].drop_duplicates()
        if not np.isnan(imp) and len(tur_row) > 0:
            cw = tur_row['tur_currency_weakness'].values[0]
            if not np.isnan(cw):
                group = df[df['country']==c]['market_group'].values[0]
                results_h4.append({'country': c, 'shock_year': sy,
                                   'impact': imp, 'tur_currency_weakness': cw, 'group': group})

res_h4 = pd.DataFrame(results_h4)
print(f"n = {len(res_h4)} country × shock-year observations")
print("\nTurkey lira YoY weakness by shock year:")
print(res_h4.groupby('shock_year')['tur_currency_weakness'].first().rename('YoY % change'))

r4, p4 = stats.pearsonr(res_h4['tur_currency_weakness'], res_h4['impact'])
print(f"\nPearson r = {r4:.3f},  p = {p4:.4f} → {'SIGNIFICANT ✓' if p4<0.05 else 'not significant'}")
print(f"Interpretation: {'negative r → larger weakness → larger drops' if r4 < 0 else 'positive r — opposite of expected'}")
print("\nCAVEAT: only 3 distinct lira-weakness values (one per shock year) — the Pearson r")
print("reflects year-level differences rather than a continuous lira-visitor relationship.")

In [ ]:
display(Image(filename=str(IMG_DIR / 'h4_lira_weakness.png'), width=900))

**Interpretation:** The significant negative Pearson r (r = −0.80, p < 0.001 per README) shows that lira weakness co-varies with visitor drops across these three shock years. **However, the critical caveat is that there are only 3 distinct lira-weakness values** — one per shock year — so the correlation is driven almost entirely by the year-to-year shift in both currency weakness and aggregate visitor drops. It cannot be interpreted as evidence that a 1% lira depreciation causes a specific % drop in visitors. The result is statistically significant but structurally fragile.

### H5 — Does Source Country GDP Per Capita Predict Visitor Volume?

**Null hypothesis (H₀):** Source country GDP per capita is not correlated with visitor volume to Turkey.

**Alternative (H₁):** Wealthier source countries send more visitors.

**Test:** Pearson + Spearman correlation (aggregate across all 391 country-year observations), plus per-group Spearman breakdown to check for Simpson's paradox — where within-group relationships might have different signs than the aggregate.

In [ ]:
# H5: GDP per capita vs visitor volume
df_h5 = df.dropna(subset=['gdp_per_capita', 'log_visitors'])
r5,  p5  = stats.pearsonr(df_h5['gdp_per_capita'], df_h5['log_visitors'])
r5s, p5s = stats.spearmanr(df_h5['gdp_per_capita'], df_h5['log_visitors'])

print(f"AGGREGATE (n={len(df_h5)} country-year obs.):")
print(f"  Pearson  r = {r5:.3f},  p = {p5:.6f}")
print(f"  Spearman r = {r5s:.3f},  p = {p5s:.6f} → {'SIGNIFICANT ✓' if p5s<0.05 else 'not significant'}")

print("\nPER-GROUP Spearman r (* = significant at α=0.05):")
group_spearman_h5 = {}
for g in ['Western Europe','Eastern Europe','Former Soviet','MENA','Other']:
    sub = df_h5[df_h5['market_group']==g]
    if len(sub) >= 5:
        rg, pg = stats.spearmanr(sub['gdp_per_capita'], sub['log_visitors'])
        group_spearman_h5[g] = (rg, pg, len(sub))
        sig_str = 'SIGNIFICANT *' if pg < 0.05 else 'not significant'
        print(f"  {g:<22}: r={rg:+.3f}, p={pg:.4f}  ({sig_str}, n={len(sub)})")

In [ ]:
display(Image(filename=str(IMG_DIR / 'h5_gdp_visitors.png'), width=900))

**Interpretation:** The aggregate Spearman is non-significant (r = −0.07, p = 0.17 per README), which at first seems counterintuitive — shouldn't wealthier countries send more tourists? But the per-group breakdown reveals a **Simpson's paradox**: Former Soviet markets show strongly positive within-group r (+0.72, significant), while MENA markets show strongly negative within-group r (−0.62, significant). The MENA pattern makes sense economically — the wealthiest MENA markets are oil-rich Gulf states (UAE, Qatar, Israel) that do send visitors, but conflict-affected markets (Iran, Syria, Iraq) are also economically constrained by sanctions and war, independently of GDP. When you pool these divergent patterns, the aggregate correlation washes out.

### H6 — Post-2023 MENA Tensions: Did They Hit MENA-Origin Markets More?

**Null hypothesis (H₀):** Post-2023 visitor changes are the same for MENA-origin and non-MENA markets (relative to 2017–2019 baseline).

**Alternative (H₁):** MENA-origin markets declined more (or grew less) than non-MENA.

**Test:** Levene's → independent t-test or Mann-Whitney U.

**Bonus descriptive split:** I also compare within-MENA subgroups — conflict-affected (Iran, Iraq, Israel, Syria) vs stable Gulf markets (UAE, Qatar). This is NOT a formal hypothesis test — n=2 in the stable subgroup is too small for any meaningful inferential test. It is purely descriptive, to help interpret whether the H6 result (if significant) reflects regional spillover or home-country conflict damage.

In [ ]:
# H6: Post-2023 MENA tensions
post_2023   = (df[df['year'].between(2023,2025)]
               .groupby(['country','market_group'])['visitors'].mean())
base_1719   = df[df['year'].between(2017,2019)].groupby('country')['visitors'].mean()
h6_df       = ((post_2023 - base_1719) / base_1719 * 100).reset_index()
h6_df.columns = ['country','market_group','pct_vs_baseline']
h6_df.dropna(subset=['pct_vs_baseline'], inplace=True)

mena_h6    = h6_df[h6_df['market_group']=='MENA']['pct_vs_baseline'].values
nonmena_h6 = h6_df[h6_df['market_group']!='MENA']['pct_vs_baseline'].values

print("H6 — Levene + t-test / Mann-Whitney U:")
h6_result = levene_then_test(mena_h6, nonmena_h6, label_a='MENA', label_b='Non-MENA')

# Within-MENA descriptive
MENA_CONFLICT = ['Iran','Iraq','Israel','Syria']
MENA_STABLE   = ['United Arab Emirates','Qatar']
mena_cf_vals  = h6_df[h6_df['country'].isin(MENA_CONFLICT)]['pct_vs_baseline']
mena_st_vals  = h6_df[h6_df['country'].isin(MENA_STABLE)]['pct_vs_baseline']

print(f"\nWithin-MENA descriptive (NOT a formal test — n=2 in stable subgroup):")
print(f"  Conflict-affected (Iran, Iraq, Israel, Syria): mean {mena_cf_vals.mean():+.1f}%  (n={len(mena_cf_vals)})")
print(f"  Stable Gulf (UAE, Qatar):                      mean {mena_st_vals.mean():+.1f}%  (n={len(mena_st_vals)})")

In [ ]:
display(Image(filename=str(IMG_DIR / 'h6_mena_tension_recent.png'), width=950))

**Interpretation:** The H6 formal test is not significant (t = −0.48, p = 0.64 per README), meaning we cannot claim that MENA-origin markets collectively performed worse than non-MENA markets in 2023–2025. However, the within-MENA descriptive split tells the more nuanced story: **conflict-affected MENA markets averaged only +1.4% above their 2017–19 baseline, while stable Gulf markets (UAE, Qatar) averaged +60%**. The aggregate MENA mean is pulled upward by the strong UAE/Qatar performance, masking the depression of conflict-affected markets. This is arguably the most actionable finding in the project: the regional MENA tension signal is real, but it affects markets according to their own political stability — not by regional proximity alone.

## 5. Machine Learning (Phases 4)

The ML section has four parts, each building on the hypothesis test findings:

1. **Linear Regression** — Can we predict log(visitors) from macro and shock features? (Temporal split: train ≤ 2019, test 2020–2025)
2. **Classification** — Can pre-COVID features identify which markets would be COVID-resilient? (LOOCV, n=17)
3. **Clustering** — Do markets naturally group by shock-response profile? (k-means k=3 + hierarchical dendrogram)
4. **Scenario 2026** — What does the pre-2020 model expect from 2025 macro conditions?

All ML code follows lecture-taught methods (Weeks 8a–11a). Panel fixed effects, ARIMA, and neural networks are deliberately excluded as they are outside lecture scope.

### 5.1 Missing Data Imputation (Week 8b)

Before running any model, I document every source of missingness and apply defensible imputation strategies. The raw `panel_dataset.csv` is never modified; imputation is applied to a working copy.

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
    classification_report, roc_curve, auc)
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage

# ── Working copy with imputation ──────────────────────────────────────────
df_ml = df.copy()

# Syria gdp_growth / gdp_per_capita: forward-fill from last pre-war value (2010)
for col in ['gdp_growth', 'gdp_per_capita']:
    df_ml[col] = df_ml.groupby('country')[col].transform(lambda s: s.ffill())

# inflation: Syria forward-fill; Iraq 2003-04 backward-fill
df_ml['inflation'] = df_ml.groupby('country')['inflation'].transform(
    lambda s: s.ffill().bfill())

# political_stability: 2025 forward-fill from 2024 (WGI not yet published for 2025)
for col in ['political_stability', 'tur_political_stability']:
    df_ml[col] = df_ml.groupby('country')[col].transform(lambda s: s.ffill())

# tur_currency_weakness 2003: no prior year to compute YoY → impute 0
df_ml['tur_currency_weakness'] = df_ml['tur_currency_weakness'].fillna(0.0)

remaining = df_ml[['gdp_growth','gdp_per_capita','inflation',
                    'political_stability','tur_currency_weakness']].isnull().sum()
print("Remaining nulls after imputation:")
print(remaining[remaining > 0].to_string() if remaining.any() else "  none — clean!")

### 5.2 Linear Regression — Predicting log(Visitors) with Temporal Split (Week 9c)

**Research question:** Can a linear model trained on pre-2020 visitor patterns predict 2020–2025 outcomes?

**Design choice:** I use a strict temporal split — train ≤ 2019, test 2020–2025. This avoids data leakage and makes "did the model anticipate COVID" a legitimate generalization test. A negative test R² is expected and honest: COVID, the Russia-Ukraine war, and MENA tensions are structurally unlike anything in the 2003–2019 training window. I should NOT tune to improve test R².

In [ ]:
# Linear Regression — temporal split
CONTINUOUS = ['gdp_growth','gdp_per_capita','inflation',
              'political_stability','tur_currency_weakness','tur_political_stability']
SHOCKS     = ['covid','coup_2016','syria_conflict','russia_turkey_crisis',
              'russia_ukraine_war','mena_tension_recent']
TARGET     = 'log_visitors'

df_reg = df_ml.copy()
grp_dummies = pd.get_dummies(df_reg['market_group'], prefix='grp', drop_first=True)
df_reg = pd.concat([df_reg, grp_dummies], axis=1)
GRP_COLS = grp_dummies.columns.tolist()
ALL_FEAT = CONTINUOUS + SHOCKS + GRP_COLS
df_reg   = df_reg.dropna(subset=ALL_FEAT + [TARGET])

train_r = df_reg[df_reg['year'] <= 2019]
test_r  = df_reg[df_reg['year'] >= 2020]
print(f"Train: {len(train_r)} rows ({train_r['year'].min()}-{train_r['year'].max()})")
print(f"Test:  {len(test_r)} rows ({test_r['year'].min()}-{test_r['year'].max()})")

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_r[ALL_FEAT])
X_test  = scaler.transform(test_r[ALL_FEAT])
y_train = train_r[TARGET].values
y_test  = test_r[TARGET].values

model_linreg = LinearRegression()
model_linreg.fit(X_train, y_train)
y_pred_train = model_linreg.predict(X_train)
y_pred_test  = model_linreg.predict(X_test)

tr_r2  = r2_score(y_train, y_pred_train)
te_r2  = r2_score(y_test,  y_pred_test)
print(f"\nTrain R² = {tr_r2:.3f}   |   Test R² = {te_r2:.3f}")
print(f"Test RMSE = {np.sqrt(mean_squared_error(y_test, y_pred_test)):.3f}")

In [ ]:
# Display regression figures
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
residuals   = y_test - y_pred_test
groups_test = test_r['market_group'].values

for g in GROUP_COLORS:
    mask = groups_test == g
    if mask.any():
        axes[0].scatter(y_pred_test[mask], residuals[mask],
                        color=GROUP_COLORS[g], alpha=0.7, s=60, label=g)
axes[0].axhline(0, color='black', lw=1.5, ls='--', alpha=0.7)
axes[0].set_xlabel('Predicted log(Visitors)')
axes[0].set_ylabel('Residual')
axes[0].set_title(f'Residual Plot (Test Set, R²={te_r2:.3f})', fontweight='bold')
axes[0].legend(fontsize=8)

for g in GROUP_COLORS:
    mask = groups_test == g
    if mask.any():
        axes[1].scatter(y_test[mask], y_pred_test[mask],
                        color=GROUP_COLORS[g], alpha=0.7, s=60, label=g)
lo = min(y_test.min(), y_pred_test.min()) - 0.3
hi = max(y_test.max(), y_pred_test.max()) + 0.3
axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1.5, alpha=0.6, label='Perfect fit')
axes[1].set_xlabel('Actual log(Visitors)')
axes[1].set_ylabel('Predicted log(Visitors)')
axes[1].set_title(f'Predicted vs Actual (Test Set)', fontweight='bold')
axes[1].legend(fontsize=8)

plt.suptitle(f'Linear Regression — Temporal Split (Train R²={tr_r2:.3f}, Test R²={te_r2:.3f})',
             fontweight='bold')
plt.tight_layout()
plt.show()

if te_r2 < 0:
    print("\n⚠ Negative Test R²: the model performs worse than predicting the mean.")
    print("This is EXPECTED and HONEST — COVID, the Russia-Ukraine war, and MENA tensions")
    print("are structurally unlike anything in the 2003-2019 training window.")
    print("The temporal split is the correct methodological choice; do not tune to fix this.")

**Why is Test R² negative?** A negative R² means the model's predictions are worse than simply using the training mean as the forecast. This is the expected and honest outcome of the temporal split. COVID (2020–2021) caused visitor volumes to collapse by 60–90% in a single year — far outside the range of any pre-2020 variation the model was trained on. Additionally, the shock dummies for `covid`, `russia_ukraine_war`, and `mena_tension_recent` all have **zero variance in the training data** (they never fire for years ≤ 2019), so OLS assigns them near-zero coefficients. Setting those dummies to 1 in the test set does nothing to the prediction. This is not a failure of the model; it is an honest reflection of the structural break.

### 5.3 Classification — COVID Resilience (Weeks 9a/9b/10)

**Research question:** Can pre-COVID market characteristics predict which countries would recover quickly from COVID?

**Design:** Binary label — "resilient" if 2022 visitors ≥ 2019 visitors (fully recovered within 2 years). Features use 2015–2019 averages only — no COVID-era data bleeds into the classifier. Leave-one-country-out cross-validation (LOOCV) is the right choice here: with only 17 countries, standard k-fold would put 3–4 countries per fold, giving unreliable held-out estimates.

In [ ]:
# Classification — COVID resilience, LOOCV
visitors_2019 = df[df['year']==2019].set_index('country')['visitors']
visitors_2022 = df[df['year']==2022].set_index('country')['visitors']
label_df = pd.DataFrame({'v2019': visitors_2019, 'v2022': visitors_2022}).dropna()
label_df['resilient'] = (label_df['v2022'] >= label_df['v2019']).astype(int)

pre_covid = df_ml[df_ml['year'].between(2015,2019)]
feat_macro = pre_covid.groupby('country')[
    ['gdp_growth','gdp_per_capita','inflation','political_stability']].mean()
feat_macro['log_baseline'] = pre_covid.groupby('country')['visitors'].mean().apply(np.log)

country_meta = (df[['country','market_group','is_mena_stable','is_mena_conflict']]
                .drop_duplicates('country').set_index('country'))
feat_macro = feat_macro.join(country_meta)
grp_dum_clf = pd.get_dummies(feat_macro['market_group'], prefix='grp', drop_first=True)
feat_matrix = feat_macro.drop(columns=['market_group']).join(grp_dum_clf)

common_c = label_df.index.intersection(feat_matrix.index)
X_clf = feat_matrix.loc[common_c].values.astype(float)
y_clf = label_df.loc[common_c, 'resilient'].values
n_clf = len(common_c)
n_res = y_clf.sum()

print(f"Dataset: {n_clf} countries × {X_clf.shape[1]} features")
print(f"Class balance: {n_res} resilient / {n_clf-n_res} non-resilient")

# LOOCV
def run_loocv(clf_factory, X, y):
    preds, probs = [], []
    for i in range(len(y)):
        X_tr = np.delete(X, i, axis=0); y_tr = np.delete(y, i)
        sc = StandardScaler(); X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X[i:i+1])
        clf = clf_factory(); clf.fit(X_tr_s, y_tr)
        preds.append(clf.predict(X_te_s)[0])
        probs.append(clf.predict_proba(X_te_s)[0][1] if hasattr(clf,'predict_proba') else np.nan)
    return np.array(preds), np.array(probs)

pred_lr, prob_lr = run_loocv(lambda: LogisticRegression(max_iter=2000,random_state=42), X_clf, y_clf)
pred_rf, prob_rf = run_loocv(lambda: RandomForestClassifier(n_estimators=200,random_state=42), X_clf, y_clf)

lr_acc = (pred_lr == y_clf).mean()
rf_acc = (pred_rf == y_clf).mean()
print(f"\nLOOCV Accuracy — Logistic Regression: {lr_acc:.2f}   Random Forest: {rf_acc:.2f}")
print("(n=17 — treat as descriptive/exploratory only, not statistically superior)")

In [ ]:
# Feature importance (RF trained on all 17)
sc_full = StandardScaler()
rf_full = RandomForestClassifier(n_estimators=200, random_state=42)
rf_full.fit(sc_full.fit_transform(X_clf), y_clf)

feat_names_clf = feat_matrix.columns.tolist()
imp_df = pd.DataFrame({'feature': feat_names_clf,
                        'importance': rf_full.feature_importances_}).sort_values('importance')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance
axes[0].barh(imp_df['feature'], imp_df['importance'], color='#9C27B0', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Mean Decrease in Impurity')
axes[0].set_title(f'Feature Importance (RF, n=17, LOOCV acc={rf_acc:.0%})', fontweight='bold')

# ROC curves
for name_roc, probs, color in [('Logistic Reg.', prob_lr, '#2196F3'), ('Random Forest', prob_rf, '#FF5722')]:
    fpr, tpr, _ = roc_curve(y_clf, probs)
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f'{name_roc} (AUC={auc(fpr,tpr):.2f})')
axes[1].plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random (0.50)')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (LOOCV, n=17 — coarse; AUC approximate)', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('COVID Resilience Classifier — LOOCV (n=17; results are exploratory)',
             fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:** Baseline market size (log mean visitors 2015–2019) and political stability are the most informative pre-COVID features for predicting resilience. This makes intuitive sense: large markets with stable political environments have more structural inertia — they bounce back faster because the underlying demand drivers are stronger. Random Forest slightly outperforms Logistic Regression (0.71 vs 0.65 accuracy), but with n=17 this difference is not statistically meaningful — both classifiers have high variance in their LOOCV estimates.

### 5.4 Clustering — Grouping Markets by Shock-Response Profile (Week 11a)

**Research question:** Do the 17 markets naturally cluster by how they respond to shocks, independently of their geographic group label?

**Design:** Each country is represented by a 5-dimensional feature vector: coup drop (%), COVID drop (%), 2023–25 % vs 2017–19 baseline, years to recover from COVID (capped at 5 = not recovered), and log mean baseline visitors. k-means with k=3 is chosen based on the elbow plot, and confirmed by hierarchical clustering.

In [ ]:
# Clustering — k-means + hierarchical
clust_rows = []
for c in all_countries:
    drop_2016 = shock_impact(c, 2016, pre_window=2)
    drop_2020 = shock_impact(c, 2020, pre_window=2)
    base_1719 = df[(df['country']==c) & df['year'].between(2017,2019)]['visitors'].mean()
    mean_2325 = df[(df['country']==c) & df['year'].between(2023,2025)]['visitors'].mean()
    pct_2325  = (mean_2325/base_1719-1)*100 if base_1719 > 0 else np.nan
    rec = years_to_recover(c, 2020, pre_window=2)
    rec_cap = min(rec, 5) if not np.isnan(rec) else 5
    log_base = np.log(base_1719) if base_1719 > 0 else np.nan
    group = df[df['country']==c]['market_group'].values[0]
    clust_rows.append({'country': c, 'market_group': group,
                       'drop_2016': drop_2016, 'drop_2020': drop_2020,
                       'pct_vs_base_2325': pct_2325, 'years_to_recover': rec_cap,
                       'log_base_visitors': log_base})

clust_df = pd.DataFrame(clust_rows).set_index('country')
# Impute Syria 2016 drop with MENA group mean
mena_mean_2016 = clust_df[clust_df['market_group']=='MENA']['drop_2016'].mean()
clust_df['drop_2016'] = clust_df['drop_2016'].fillna(mena_mean_2016)

feat_cols = ['drop_2016','drop_2020','pct_vs_base_2325','years_to_recover','log_base_visitors']
X_clust_s = StandardScaler().fit_transform(clust_df[feat_cols].astype(float))

# Elbow
inertias = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_clust_s).inertia_ for k in range(2,9)]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(2,9), inertias, marker='o', color='#2196F3', lw=2.5, ms=8)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Plot — Choose k', fontweight='bold')
axes[0].axvline(3, color='red', lw=1.5, ls='--', alpha=0.7, label='Chosen k=3')
axes[0].legend(fontsize=9)

# k=3 assignments
km3 = KMeans(n_clusters=3, n_init=10, random_state=42)
km3.fit(X_clust_s)
clust_df['cluster'] = km3.labels_

colors_clust = [GROUP_COLORS[g] for g in clust_df['market_group']]
for cl in sorted(clust_df['cluster'].unique()):
    members = clust_df[clust_df['cluster']==cl]
    axes[1].scatter([cl]*len(members),
                    members['pct_vs_base_2325'],
                    c=[GROUP_COLORS[g] for g in members['market_group']],
                    s=100, alpha=0.8, edgecolors='black', lw=0.5, zorder=3)
    for country, row in members.iterrows():
        axes[1].annotate(country[:5], (cl, row['pct_vs_base_2325']),
                         fontsize=7.5, xytext=(5,0), textcoords='offset points')
axes[1].set_xticks([0,1,2]); axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('2023-25 % vs 2017-19 baseline')
axes[1].set_title('Cluster Assignments by 2023-25 Performance', fontweight='bold')

legend_patches = [mpatches.Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
axes[1].legend(handles=legend_patches, fontsize=8)
plt.suptitle('k-Means Clustering (k=3) — Market Groups by Shock-Response Profile', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCluster assignments:")
for cl in sorted(clust_df['cluster'].unique()):
    members = clust_df[clust_df['cluster']==cl]
    print(f"  Cluster {cl}: {', '.join(members.index.tolist())}")

In [ ]:
# Hierarchical dendrogram
Z = linkage(X_clust_s, method='ward')
fig, ax = plt.subplots(figsize=(14, 6))
country_map = df.set_index('country')['market_group'].to_dict()
dendrogram(Z, labels=clust_df.index.tolist(), ax=ax,
           color_threshold=0.7*max(Z[:,2]), leaf_rotation=45, leaf_font_size=10)
for lbl in ax.get_xticklabels():
    if lbl.get_text() in country_map:
        lbl.set_color(GROUP_COLORS.get(country_map[lbl.get_text()], 'black'))
ax.set_ylabel('Ward Linkage Distance')
ax.set_title('Hierarchical Clustering Dendrogram (label color = market group)', fontweight='bold')
legend_patches = [mpatches.Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=legend_patches, fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

**Interpretation:** The k=3 clustering produces three meaningfully distinct groups. The elbow plot supports k=3 (inertia drop slows after that). **Critically, UAE and Qatar separate from the conflict-affected MENA markets** (Iran, Iraq, Israel, Syria) — they cluster with fast-recovering or high-growth markets. This aligns directly with the H6 within-MENA descriptive finding: Gulf-stable markets have a fundamentally different shock-response profile than conflict-affected MENA markets. The clustering finds this pattern without being told about geographic or political categories.

### 5.5 Scenario 2026 — What Does the Model Expect Under Continued MENA Tensions?

The final ML step uses the linear regression from Section 5.2 to build a structural baseline: what visitor volumes would the pre-COVID model expect for 2025–2026 conditions, with `mena_tension_recent=1`?

**Important limitation:** Because `mena_tension_recent` has zero variance in the training data (it never fires for years ≤ 2019), its coefficient is near-zero (~1e-17, floating-point noise). Setting it to 1 does not change the prediction. What the scenario actually shows is the structural gap between the pre-COVID model's expectations and 2025 actual visitor levels — driven by GDP per capita and market group coefficients, not the MENA tension variable.

In [ ]:
display(Image(filename=str(IMG_DIR / 'ml_06_scenario_2026.png'), width=900))

**Reading the scenario chart:** Each pair of bars shows actual 2025 visitors (faded) vs the model's projection (outlined). The gap between the two reflects how far 2025 reality diverges from pre-COVID structural expectations. Markets where actual 2025 visitors *exceed* the model's expectation (bar pairs where projected is lower than actual) have overperformed relative to pre-COVID macro trends — these tend to be Gulf markets and some Western European markets. Markets where the model's projection *exceeds* actual (e.g., Russia, Syria) are structurally displaced by post-2019 geopolitical breaks that the pre-2020 model could not have anticipated.

## 6. Conclusions & Actionable Recommendations

### What the hypothesis tests actually showed

The six hypothesis tests produced a mixed but coherent picture. Only H4 (Turkey lira weakness) returned a statistically significant Pearson correlation (r = −0.80, p < 0.001) — but with the important caveat that this is driven by only three distinct shock years, not a continuous lira-visitor relationship. H1, H2, H3, H5, and H6 were all non-significant at α = 0.05. This honesty matters: the instinct to frame every descriptive pattern as a confirmed finding would be statistically invalid. The large within-group variance across all 17 markets, combined with the small effective sample sizes per group, limits inferential power throughout.

That said, non-significant aggregate tests do not mean the findings are meaningless. The per-group Spearman breakdown in H5 revealed a genuine Simpson's paradox — Former Soviet markets show a strongly positive GDP-visitor relationship (+0.72) while MENA markets show a strongly negative one (−0.62), effects that cancel in aggregate. The H6 within-MENA descriptive split found that stable Gulf markets (UAE, Qatar) averaged +60% above their 2017–19 baseline in 2023–2025, while conflict-affected MENA markets averaged only +1.4%. These are important findings that don't require a statistically significant aggregate test to be actionable.

### What the ML clustering revealed

The k-means clustering (k=3) successfully separated the 17 markets into three distinct shock-response profiles without being given geographic or political labels. The most important finding: UAE and Qatar cluster with fast-recovering markets, not with Iran, Iraq, Israel, and Syria. This data-driven separation confirms what the H6 descriptive analysis suggested — that "MENA" as a geographic label conflates two categorically different types of markets.

### Actionable recommendations for Turkish tourism firms

Given rising MENA tensions, the data strongly suggests that **Turkish tourism firms should prioritize UAE and Qatar** as growth targets. These are large, wealthy, politically stable Gulf markets that are already showing strong above-baseline growth (+40-80% vs 2017–19). They are not suppressed by home-country conflict, and their positive trajectory is distinct from regional spillover effects. By contrast, conflict-affected MENA markets (Iran, Iraq, Israel, Syria) face structural suppression that is unlikely to resolve without geopolitical change — firms should maintain presence but not invest heavily in market development there in the near term.

For Western Europe, the data shows these markets are resilient and have recovered well, but growth is slow. They remain the largest source of visitors and should anchor the portfolio, not be deprioritized. Former Soviet markets have been structurally disrupted by the Russia-Ukraine war — Russia's displacement is likely permanent under current sanctions, but Georgia and Azerbaijan show positive trajectories and may be underserved.

### Limitations

This analysis faces four key limitations. First, n=17 countries is a small panel for inferential claims about group-level differences; most results are descriptive with insufficient power to detect moderate effect sizes. Second, the temporal split exposes the linear regression to three major structural breaks (COVID, Russia-Ukraine war, MENA tensions) that lie entirely outside its training distribution — the negative test R² is expected, not a modelling failure, but it means the scenario forecasts are not reliable as point estimates. Third, Syria's near-zero visitor counts from 2011 onward create a floor effect that influences group averages for MENA. Fourth, UAE-Turkey bilateral relations had a notable chill from 2017–2021 (Qatar blockade dispute) that is not captured by a dedicated dummy variable; this may slightly suppress UAE coefficients in the linear model.

### Future work

The most valuable extension would be adding post-2023 training data as it accumulates, which would give the MENA tension dummy a non-zero coefficient and enable genuine impact estimation. A difference-in-differences design comparing MENA-stable vs MENA-conflict markets before and after 2023 would cleanly isolate the regional spillover effect from home-country damage — and would be a legitimate follow-on to the H6 descriptive split done here.